# Phân loại Spam/Ham bằng viBERT

**Nhóm Bảo, Cương, Phi, Phương**

Fine-tune viBERT cho bình luận PTIT, lựa chọn bằng validation và đối chiếu với
TF-IDF + SVM trên cùng tập test. Notebook trình bày dữ liệu, tokenization,
huấn luyện, đánh giá, phân tích lỗi và ứng dụng mô hình đã lưu.


## 1. Cấu hình môi trường

Colab sử dụng GPU và thư viện trong `requirements/modern-colab.txt`; môi trường
local dùng `.venv-modern` và `requirements/modern.txt`. Code, v7 và kết quả S3
được đặt trong cùng project theo hướng dẫn `docs/guides/HUONG_DAN_VIBERT_COLAB.md`.

`MODE="review"` đọc kết quả và checkpoint đã huấn luyện. `MODE="train"` thực
hiện fine-tune, chọn cấu hình, huấn luyện cuối và đánh giá trước khi trình bày.
Các cell đọc bảng/hình không tự huấn luyện lại. Chạy theo thứ tự từ đầu.


In [ ]:
import os, sys, json
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "pipeline/modern/train.py").exists()), Path("/content/ptit-facebook-spam-ham"))
if not (PROJECT_ROOT / "pipeline/modern/train.py").exists():
    raise FileNotFoundError("Chuẩn bị gói project theo hướng dẫn viBERT trước khi chạy notebook.")
sys.path.insert(0, str(PROJECT_ROOT))
MODE = "review"
CSV_PATH = PROJECT_ROOT / "data/final/labeled_v7.csv"
CONFIG_PATH = PROJECT_ROOT / "pipeline/modern/vibert_config.json"
OUTPUT_DIR = Path(os.environ.get("SPAMHAM_VIBERT_OUTPUT", str(PROJECT_ROOT / "artifacts/modern/vibert")))
MODEL_DIR = OUTPUT_DIR / "model"
BASELINE_PREDICTIONS = PROJECT_ROOT / "artifacts/spamham/test_predictions.csv"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, transformers
from transformers import AutoTokenizer
from IPython.display import Markdown, Image
from pipeline.presentation.notebook_display import display_readable as display, code_excerpt_html
from pipeline.modern.data import prepare_data, split_summary
from pipeline.modern.train import run_experiment
from pipeline.modern.predict import VibertPredictor
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)
print("Thiết bị:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Chế độ:", MODE, "| Kết quả:", OUTPUT_DIR)


## 2. Dữ liệu và thiết kế thực nghiệm

Giữ v7 và quy tắc nhóm văn bản của Notebook 02. Tập test gồm cùng 7.474 ID.
Trong train, chọn fold đầu làm validation; các nhóm văn bản không giao nhau.

Huấn luyện phát triển trên 24.815 mẫu, kiểm định trên 5.944 mẫu. Chọn loại loss
và epoch theo **F1 spam trên validation**. Sau đó khởi tạo lại từ viBERT gốc,
huấn luyện cấu hình đã chọn trên toàn bộ 30.759 mẫu train với số epoch đã xác
định; chỉ lúc này mới đánh giá test. Điểm validation không phải CV F1 năm fold.


In [ ]:
data = prepare_data(CSV_PATH, CONFIG["validation_fold"])
display(split_summary(data))
preview = data[data.development_split.eq("fit")].groupby("label").sample(n=3, random_state=CONFIG["seed"])
display(preview[["id", "label", "text", "model_text"]])


## 3. Chuẩn hóa và tokenization

Chuẩn hóa Unicode, HTML và khoảng trắng, giữ chữ hoa/thường và từ dừng. Tokenizer
cased của viBERT tách subword, bổ sung token đặc biệt và tạo attention mask.
Padding theo batch, cắt các chuỗi vượt độ dài cấu hình. Bảng token dùng bình luận
trong train; tỷ lệ bị cắt được khảo sát trên train, không dùng test để chọn độ dài.


In [ ]:
tokenizer_source = MODEL_DIR if MODEL_DIR.exists() else CONFIG["model_name"]
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source,
    **({"local_files_only": True} if MODEL_DIR.exists() else {"revision": CONFIG["revision"], "token": False}),
    do_lower_case=False)
token_rows = []
for row in preview.itertuples():
    ids = tokenizer(row.model_text, truncation=True, max_length=CONFIG["max_length"])["input_ids"]
    token_rows.append({"ID": row.id, "Bình luận": row.model_text, "Số token sau cắt": len(ids),
                       "Token đầu": " ".join(tokenizer.convert_ids_to_tokens(ids[:24]))})
display(pd.DataFrame(token_rows))
train_text = data.loc[data.outer_split.eq("train"), "model_text"].tolist()
lengths = pd.Series([len(ids) for ids in tokenizer(train_text, truncation=False, verbose=False)["input_ids"]])
display(pd.DataFrame({"Số bình luận train": [len(lengths)], "Giới hạn token": [CONFIG["max_length"]],
                      "Tỷ lệ vượt giới hạn (%)": [round(lengths.gt(CONFIG["max_length"]).mean()*100, 2)]}))


## 4. Xây dựng mô hình

Sử dụng checkpoint `FPTAI/vibert-base-cased`, bổ sung đầu phân loại hai nhãn:
0 = ham, 1 = spam. Fine-tune cả encoder và đầu phân loại bằng dữ liệu nhóm.
viBERT tạo biểu diễn ngữ cảnh rồi trả logits hai lớp; softmax chuyển thành
xác suất spam. Quy tắc phân loại dùng ngưỡng xác suất 0,5 cố định.

Checkpoint gốc là mô hình tiền huấn luyện, chưa phải bộ phân loại spam PTIT.
Mã nạp kiểm tra trọng số encoder; đầu phân loại được học trong fine-tune.


In [ ]:
display(code_excerpt_html(PROJECT_ROOT / "pipeline/modern/train.py", "load_pretrained", max_lines=25))
display(pd.DataFrame([{"Checkpoint": CONFIG["model_name"], "Nhãn": "ham / spam",
                       "Ngưỡng xác suất": CONFIG["threshold"], "Seed": CONFIG["seed"]}]))


## 5. Huấn luyện và lựa chọn cấu hình

Đối chiếu cross-entropy thông thường và cross-entropy có trọng số lớp. Trọng số
chỉ tính từ phần train đang dùng; validation/test giữ phân bố nhãn tự nhiên.
Đây là khảo sát bổ trợ về mất cân bằng. Learning rate, batch, độ dài và số epoch
được khai báo trong `vibert_config.json`. Mỗi lượt lưu checkpoint theo epoch để
có thể tiếp tục khi runtime ngắt; không resume sang cấu hình hoặc ID khác.

Chọn loss và epoch có validation F1 cao nhất, huấn luyện lại từ checkpoint gốc
trên toàn bộ train. Biểu đồ và bảng bên dưới được đọc từ lịch sử huấn luyện đã
ghi; không dùng chỉ số test để lựa chọn hoặc thay đổi cấu hình.


In [ ]:
if MODE == "train":
    run_experiment(CSV_PATH, OUTPUT_DIR, BASELINE_PREDICTIONS, CONFIG_PATH, stage="all")
elif MODE != "review":
    raise ValueError("MODE cần là train hoặc review.")
if not (OUTPUT_DIR / "verification.json").exists():
    raise FileNotFoundError("Chưa có lượt huấn luyện hoàn tất. Xem hướng dẫn Colab để chạy và tải kết quả.")
validation = pd.read_csv(OUTPUT_DIR / "validation_metrics.csv")
selected = json.loads((OUTPUT_DIR / "selected.json").read_text(encoding="utf-8"))
display(validation.round(4))
display(pd.Series(selected, name="Cấu hình lựa chọn").to_frame())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for variant in CONFIG["variants"]:
    history = pd.read_csv(OUTPUT_DIR / "selection" / variant / "training_history.csv")
    loss_rows = history.dropna(subset=["loss"])
    validation_rows = history.dropna(subset=["eval_f1"])
    axes[0].plot(loss_rows["step"], loss_rows["loss"], label=variant)
    axes[1].plot(validation_rows["epoch"], validation_rows["eval_f1"], marker="o", label=variant)
axes[0].set(xlabel="Bước cập nhật", ylabel="Train loss", title="Diễn biến loss")
axes[1].set(xlabel="Epoch", ylabel="Validation F1 spam", title="Lựa chọn bằng validation")
for ax in axes: ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. Đánh giá và so sánh mô hình

Đo Accuracy, Precision, Recall, F1 spam và AP trên cùng ID test với S3. Bảng này
so sánh hai pipeline hoàn chỉnh, có cách biểu diễn/chuẩn hóa phù hợp từng phương
pháp. Chỉ so sánh các chỉ số test; không đối chiếu validation F1 của viBERT với
CV F1 của SVM như hai phép đo tương đương.

Ma trận nhầm lẫn thể hiện FP/FN. Đường Precision–Recall dùng xác suất viBERT và
decision score SVM để đánh giá khả năng xếp hạng; hai loại score không cùng thang.


In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / "comparison_with_svm.csv")
display(comparison.round(4))
display(Image(filename=str(OUTPUT_DIR / "figures/confusion_vibert.png")))
display(Image(filename=str(OUTPUT_DIR / "figures/pr_comparison.png")))


## 7. Phân tích dự đoán

Phân tích lỗi của **viBERT đã chọn**, dùng ID và bình luận gốc từ test. FP là ham
bị nhận nhầm; FN là spam bị bỏ sót. Xác suất spam là đầu ra mô hình, không phải
bảo đảm độ tin cậy đã hiệu chỉnh. Các ví dụ dùng để giải thích, không tự sửa nhãn.


In [ ]:
test_predictions = pd.read_csv(OUTPUT_DIR / "test_predictions.csv", keep_default_na=False, dtype={"id": str})
expected_ids = data.loc[data.outer_split.eq("test"), "id"].tolist()
assert test_predictions.id.tolist() == expected_ids
print("Ham bị nhận nhầm thành spam (FP):")
display(test_predictions[test_predictions.label.eq("ham") & test_predictions.prediction.eq("spam")]
        .sort_values("probability_spam", ascending=False).head(8))
print("Spam bị bỏ sót (FN):")
display(test_predictions[test_predictions.label.eq("spam") & test_predictions.prediction.eq("ham")]
        .sort_values("probability_spam").head(8))


## 8. Lưu mô hình và dự đoán bình luận mới

`train.py` lưu model và tokenizer bằng `save_pretrained()` vào `model/`. Thư mục
chứa trọng số, cấu hình BERT, ánh xạ nhãn, tokenizer và cấu hình chuẩn hóa/ngưỡng.
Sau huấn luyện, nạp lại và kiểm tra dự đoán trên toàn bộ test trước khi xác nhận
hoàn tất. Demo bên dưới dùng chính thư mục này, không huấn luyện lại.


In [ ]:
verification = json.loads((OUTPUT_DIR / "verification.json").read_text(encoding="utf-8"))
display(pd.Series({"Trạng thái": verification["status"], "Số mẫu test": verification["test_rows"],
                   "Nạp lại khớp dự đoán": verification["reload_predictions_match"],
                   "Ngưỡng xác suất": verification["threshold"]}, name="Kiểm tra").to_frame())
predictor = VibertPredictor(MODEL_DIR)
display(predictor.predict(["Cho mình hỏi lịch thi với ạ", "Liên hệ 0901234567 hoặc https://example.com",
                           "Có bạn nào học môn này không?", "."]))


## 9. Nhận xét

Nhận xét dựa trên kết quả test đã đo, cùng giới hạn dữ liệu và phạm vi lựa chọn.
Các câu demo minh họa cách sử dụng, không thay thế đánh giá trên test.


In [ ]:
modern = comparison[comparison["Mã"].eq("M1")].iloc[0]
svm = comparison[comparison["Mã"].eq("S3")].iloc[0]
fp = int((test_predictions.label.eq("ham") & test_predictions.prediction.eq("spam")).sum())
fn = int((test_predictions.label.eq("spam") & test_predictions.prediction.eq("ham")).sum())
display(Markdown(f"""
- **Lựa chọn:** viBERT với loss `{selected['variant']}`, {selected['epoch']} epoch, chọn bằng validation F1.
- **Test:** Accuracy {modern['Accuracy']:.2%}, Precision {modern['Precision']:.2%},
  Recall {modern['Recall']:.2%}, F1 {modern['F1']:.4f}, AP {modern['AP']:.4f}.
  Có {fp:,} ham bị nhận nhầm và {fn:,} spam bị bỏ sót.
- **Đối chiếu S3:** chênh lệch F1 = {modern['F1']-svm['F1']:+.4f}; đọc cùng Precision/Recall và loại lỗi.
- **Giới hạn:** kết quả phụ thuộc nhãn hiện có, dữ liệu gần trùng, bình luận ngắn và ngữ cảnh bài đăng.
  Chưa khảo sát nhiều seed hoặc mọi cấu hình; không khẳng định một mô hình luôn tốt hơn.
- **Ứng dụng:** checkpoint đã lưu dùng để nạp lại và phân loại bình luận mới bằng cùng tokenizer/ngưỡng.
"""))


## Tài liệu tham khảo

- viBERT: https://github.com/fpt-corp/viBERT
- Checkpoint: https://huggingface.co/FPTAI/vibert-base-cased
- Transformers — text classification: https://huggingface.co/docs/transformers/tasks/sequence_classification
- PyTorch — CrossEntropyLoss: https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
- Dataset PTIT: https://huggingface.co/datasets/p-storm/ptit-facebook-comments-spamham
